# SōmaGraph — Pose Engine 実証ノートブック

AI相馬眼プラットフォーム **SōmaGraph** の解析エンジン中核を、実際に動かして検証する。

このノートブックがやること:
1. SuperAnimal-Quadruped（馬対応の学習済みポーズ推定モデル）を導入
2. アップロードした馬の動画から、全フレームの関節を**自動検出**（手打ち座標ゼロ）
3. 関節座標の時系列から、SōmaGraph独自の**歩様スコア・左右差・関節可動域**を計算
4. ラベル付き動画 + 数値レポート（JSON）を出力

> モデル: DeepLabCut Model Zoo `superanimal_quadruped`（EPFL Mathis Lab, 研究用途/非商用）。
> Horse-10 等で評価された四足動物39関節モデル。**追加学習なし(zero-shot)**で馬に使える。

---
### 使い方
1. ランタイム → ランタイムのタイプを変更 → **GPU(T4)** を選択
2. 上から順にセルを実行（▶）
3. 「動画アップロード」セルで、解析したい馬の横向き歩様動画を選ぶ


## 1. 環境セットアップ
DeepLabCut 3.0(PyTorchエンジン)をインストール。数分かかる。

In [ ]:
# GPUが有効か確認
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "GPU未検出 → ランタイムのタイプをGPUに変更してください")


In [ ]:
# DeepLabCut 3.0 (PyTorch) をインストール
%pip install -q "deeplabcut[pytorch]>=3.0.0"
print("インストール完了。ランタイム再起動を求められたら再起動して、ここから再実行してください。")


## 2. 動画をアップロード
横向き(side view)で馬の全身が映った歩様動画が最適。セリ動画でもOK。

In [ ]:
from google.colab import files
import os
print("解析したい馬の動画(.mp4/.mov)を選択してください...")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print("アップロード完了:", video_path, f"({os.path.getsize(video_path)/1e6:.1f} MB)")


## 3. SuperAnimalで関節を自動検出

`video_inference_superanimal` 一発で、全フレームの関節検出 → 追跡 → ラベル付き動画出力まで実行される。

- `superanimal_quadruped` : 馬を含む四足動物モデル
- `model_name=hrnet_w32` : 高精度版(top-down)
- `video_adapt=True` : この動画の明るさ/見た目に自己教師あり適応(精度向上・ジッター低減、ラベル不要)
- `pcutoff` : 信頼度しきい値。これ未満の関節は描画しない


In [ ]:
import deeplabcut

superanimal_name = "superanimal_quadruped"
model_name      = "hrnet_w32"
detector_name   = "fasterrcnn_resnet50_fpn_v2"
pcutoff         = 0.15

deeplabcut.video_inference_superanimal(
    [video_path],
    superanimal_name,
    model_name=model_name,
    detector_name=detector_name,
    videotype=os.path.splitext(video_path)[1],
    video_adapt=True,        # 動画適応ON(精度向上)
    pcutoff=pcutoff,
    pseudo_threshold=0.1,
)
print("\n検出完了。_labeled.mp4 と .h5(座標データ) が生成された。")


## 4. ラベル付き動画を確認
関節が馬の体に乗っているか目視チェック。

In [ ]:
import glob
from IPython.display import HTML
from base64 import b64encode

labeled = sorted(glob.glob("*_labeled.mp4") + glob.glob("*_full.mp4"))
print("生成された動画:", labeled)
if labeled:
    # Colabで再生(大きい場合は左のファイルタブからDLして確認)
    mp4 = open(labeled[0],"rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'<video width=560 controls><source src="{data_url}" type="video/mp4"></video>'))


## 5. SōmaGraph 解析 — 関節座標からスコアを計算

ここがSōmaGraphの独自価値。検出された関節の時系列(.h5)を読み、
**歩様の質**を数値化する。

計算する指標:
- **左右差(Lateral asymmetry)**: 左右の同じ関節の動きの振幅差。小さいほど均整がとれている
- **ストライド(歩幅)**: 蹄(球節)の前後移動の振幅
- **関節可動域(ROM)**: 飛節・膝の角度変化の大きさ → 推進力の指標
- **リズム安定性**: 歩周期のばらつき(変動係数)。小さいほど安定


In [ ]:
import pandas as pd, numpy as np, glob

h5 = sorted(glob.glob("*.h5"))
print("座標ファイル:", h5)
df = pd.read_hdf(h5[0])
# DLCのMultiIndex列を平坦化
scorer = df.columns.levels[0][0]
df = df[scorer]
bodyparts = list(df.columns.levels[0])
print(f"\n検出された関節 ({len(bodyparts)}個):")
print(", ".join(bodyparts))


In [ ]:
def series(bp, coord):
    """関節bpの座標時系列を返す(信頼度でフィルタ)"""
    if bp not in bodyparts: return None
    x = df[bp]["x"].values
    y = df[bp]["y"].values
    p = df[bp]["likelihood"].values
    mask = p > 0.5
    s = (x if coord=="x" else y).astype(float)
    s[~mask] = np.nan
    return pd.Series(s).interpolate().bfill().ffill().values

def amplitude(arr):
    """移動振幅(95-5パーセンタイル)"""
    if arr is None or np.all(np.isnan(arr)): return np.nan
    return np.nanpercentile(arr,95) - np.nanpercentile(arr,5)

# --- 左右の対応する関節ペアを、関節名から自動で見つける ---
# SuperAnimalの命名ゆれ(left_/right_, _left/_right, L_/R_ 等)に対応
def find_pairs(parts):
    pairs = []
    used = set()
    for bp in parts:
        low = bp.lower()
        if "left" in low or low.startswith("l_") or "_l_" in low:
            # 対応する right を探す
            cand = (low.replace("left","right")
                       .replace("l_","r_"))
            for other in parts:
                if other.lower()==cand and other not in used:
                    base = low.replace("left","").replace("l_","").strip("_ ")
                    pairs.append((base or bp, bp, other))
                    used.add(bp); used.add(other)
                    break
    return pairs

auto_pairs = find_pairs(bodyparts)
print("自動検出した左右ペア:")
for base, l, r in auto_pairs:
    print(f"  {base:16s}: {l}  <->  {r}")
if not auto_pairs:
    print("  (左右ペアが見つかりません。bodypartsを確認:", bodyparts, ")")


In [ ]:
# --- 左右差(歩様の均整)を計算 ---
print("=== SōmaGraph 歩様解析レポート ===\n")
report = {}
for base, lbp, rbp in auto_pairs:
    la = amplitude(series(lbp,"y"))
    ra = amplitude(series(rbp,"y"))
    if np.isnan(la) or np.isnan(ra):
        continue
    asym = abs(la-ra)/max(la,ra,1e-6)*100
    report[base] = {"left_amp":round(la,1),"right_amp":round(ra,1),"asymmetry_pct":round(asym,1)}
    print(f"{base:16s} 左振幅={la:6.1f}px 右振幅={ra:6.1f}px  左右差={asym:5.1f}%")

if not report:
    print("有効な左右ペアの振幅が取れませんでした。pcutoffを下げるか動画を長くしてください。")


In [ ]:
# --- リズム安定性: いずれかの肢(蹄/足先)の上下動の周期性 ---
import numpy as np
# 足先っぽい関節を自動選択(paw/hoof/foot/ankleを含む名前)
foot_bp = next((bp for bp in bodyparts
                if any(k in bp.lower() for k in ["paw","hoof","foot","ankle","fetlock"])), None)
print("リズム解析に使う関節:", foot_bp)
if foot_bp:
    fp = series(foot_bp,"y")
    sig = fp - np.nanmean(fp)
    crossings = np.where(np.diff(np.sign(sig)))[0]
    if len(crossings) > 2:
        periods = np.diff(crossings)
        cv = np.std(periods)/np.mean(periods)*100
        stability = max(0, 100 - cv)
        print(f"歩周期の変動係数: {cv:.1f}%")
        print(f"リズム安定スコア: {stability:.0f}/100")
    else:
        print("周期検出に十分なサイクルがありません(動画を長めに)")


## 6. 総合スコア化(SōmaGraphダッシュボード用JSON)

ダッシュボードの「総合スコア 91.2」「左右差 8%」等は、本来こうして算出される。
ここで吐くJSONを、SomaGraphフロントエンドの `/api/horse/{id}/analysis` がそのまま受け取る想定。


In [ ]:
import json

def to_score(asym_pct):
    """左右差% → 100点満点スコア(小さいほど高得点)"""
    return round(max(0, 100 - asym_pct*1.5), 1)

analysis = {
    "video": video_path,
    "model": f"{superanimal_name}/{model_name}",
    "frames_analyzed": int(len(df)),
    "lateral_asymmetry": report,
    "gait_scores": {k: to_score(v["asymmetry_pct"]) for k,v in report.items()},
}
# 総合スコア = 各部位スコアの加重平均(例)
if analysis["gait_scores"]:
    vals = list(analysis["gait_scores"].values())
    analysis["overall_score"] = round(sum(vals)/len(vals), 1)

print(json.dumps(analysis, indent=2, ensure_ascii=False))

with open("somagraph_analysis.json","w") as f:
    json.dump(analysis, f, indent=2, ensure_ascii=False)
print("\n→ somagraph_analysis.json を出力。左のファイルタブからダウンロードできる。")


---
## まとめ

ここまでで実証できたこと:
- 馬の動画から、学習済みNNが**全フレームの関節を自動検出**(手打ちゼロ・追跡付き)
- その座標から、SōmaGraphの**歩様スコア・左右差**を計算
- ダッシュボードに渡せる**JSON**を生成

### 本番(SōmaGraph)への組み込み
- このノートブックの処理を **FastAPIのワーカー**にして、動画アップロードで非同期起動
- GPU: Colab→本番はクラウドGPU(L4/A10等)。1動画(数百フレーム)で数十秒〜数分
- 関節命名がモデル版で違うので、`pairs` を実際の `bodyparts` 出力に合わせて確定する
- 精度が足りなければ、870の馬数十頭分をラベルして **fine-tune**(SuperAnimalは10倍少ないデータで足りる)

### 注意
SuperAnimalは**研究用途/非商用**ライセンス。商用化(SōmaGraphを売る)なら、
AP-10K等でYOLO-poseを自前fine-tuneするか、ライセンス取得が必要。
